# Semana 6 — Funciones, Variables Globales y Locales, y Módulos

**Física Computacional — 106018C** · Lenguajes: **Fortran** y **Python** · Cierre del Capítulo 2: Estructura del programa

Hasta la Semana 5, cada programa fue un solo bloque: datos, cálculo y salida en el mismo `program`. Cuando un cálculo se repite (tres casos de verificación, veinte corridas de Monte Carlo, ocho núcleos distintos), ese bloque se copia una y otra vez. Esta semana se estudia cómo dividir un programa en **procedimientos** (funciones y subrutinas), qué variables ve cada uno, y cómo agruparlos en **módulos**.

Cada concepto se presenta en los dos lenguajes del curso, uno al lado del otro: primero en Fortran y después en Python. Los programas de Fortran se escriben con `%%writefile`, se compilan con `gfortran` y se ejecutan desde la celda siguiente.

**Objetivos de la sesión**

1. Escribir funciones y subrutinas en Fortran, y funciones en Python, que reciban datos por argumentos y devuelvan resultados.
2. Explicar cómo se pasan los argumentos en cada lenguaje y usar `intent` para proteger los datos de entrada.
3. Distinguir variables locales y globales, y reconocer los errores típicos de cada lenguaje.
4. Organizar constantes y procedimientos en módulos (`module` en Fortran, archivo `.py` en Python).
5. Resolver un problema físico completo con un programa modular: el **Taller 6**, sobre la fórmula semiempírica de masa de los núcleos atómicos.

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

**Alcance.** Igual que en las semanas anteriores, los programas de **Fortran** no usan arreglos: estos se estudian en el Capítulo 3 (Semana 7). En Python se permite leer columnas de datos con NumPy y utilizarlas para el análisis y las gráficas. Tampoco se pasan funciones como argumento de otras funciones; el tratamiento de interfaces de procedimientos se deja para una etapa posterior del curso.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento.** Los recuadros dorados como este proponen ejercicios cortos, no calificados, que preparan el Taller 6. Cada uno dice qué debe obtener. Se recomienda resolverlos antes de la sesión del lunes.

</div>

## 0. Preparación

En Colab, `gfortran` suele estar instalado. La celda siguiente lo comprueba y lo instala si falta. También crea la carpeta `taller_nucleo_atomico/`, donde irán los archivos del Taller.

In [ ]:
# Comprueba gfortran (y lo instala en Colab si falta) y crea la carpeta del Taller
!which gfortran > /dev/null || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
!gfortran --version | head -1
!mkdir -p taller_nucleo_atomico

## 1. Por qué dividir un programa

En el Taller 1, el programa `altura_maxima` calculaba

$$h_{\max} = \frac{(v_0 \sin\theta)^2}{2g}$$

donde $v_0$ es la rapidez inicial [m/s], $\theta$ el ángulo de lanzamiento medido desde la horizontal, $v_0\sin\theta$ la componente vertical de la velocidad inicial y $g = 9.8$ m/s² la aceleración de la gravedad. Se calculaba para **un** par $(v_0, \theta)$ leído del teclado. Para verificarlo con tres casos conocidos había que ejecutarlo tres veces y escribir los datos a mano en cada una:

| $v_0$ [m/s] | $\theta$ [°] | $h_{\max}$ [m] |
|---|---|---|
| 9.8 | 90 | 4.9000 |
| 20 | 45 | 10.2041 |
| 20 | 30 | 5.1020 |

La fórmula aparecía una sola vez, pero la unidad que se podía repetir era el programa completo. Un **procedimiento** es un bloque de código con nombre que recibe datos, hace un cálculo y devuelve un resultado. Se escribe una vez y se usa (se *llama*) tantas veces como se necesite, con datos distintos.

| | Fortran | Python |
|---|---|---|
| Devuelve **un** valor | `function` | `def ... return valor` |
| Devuelve **varios** resultados | `subroutine` con argumentos de salida | `def ... return a, b, c` |
| Se llama con | `y = f(x)` / `call s(x, y, z)` | `y = f(x)` / `a, b, c = s(x)` |

## 2. Funciones

### 2.1 Fortran: `function` dentro de `contains`

Una función interna se escribe después de la instrucción `contains`, antes del `end program`. Su encabezado declara el tipo del resultado, el nombre y los **argumentos ficticios** (los nombres que usa la función por dentro). El resultado se asigna al nombre de la función.

El atributo `intent(in)` declara que el argumento solo entra: la función no puede modificarlo (sección 4).

In [ ]:
%%writefile funcion_altura.f90
! Semana 6 -- Una función en Fortran: altura máxima de un proyectil
program funcion_altura
  implicit none
  integer, parameter :: dp = kind(1.0d0)          ! reales de doble precisión

  ! --- Programa principal: tres llamadas a la misma función, con datos distintos ---
  ! La función se usa dentro de una expresión (aquí, en la lista de print),
  ! igual que sin(x) o sqrt(x).
  print '(a, f10.4)', 'v0 =  9.8, theta = 90:  h_max =', altura_maxima(9.8_dp, 90.0_dp)
  print '(a, f10.4)', 'v0 = 20.0, theta = 45:  h_max =', altura_maxima(20.0_dp, 45.0_dp)
  print '(a, f10.4)', 'v0 = 20.0, theta = 30:  h_max =', altura_maxima(20.0_dp, 30.0_dp)

! contains separa el programa principal de sus procedimientos internos.
contains

  ! ------------------------------------------------------------------
  ! altura_maxima: altura máxima de un proyectil lanzado desde el suelo
  !   Entradas: v0 [m/s], theta_grados [grados]
  !   Salida:   altura máxima [m], como valor de la función
  ! ------------------------------------------------------------------
  ! Encabezado: tipo del resultado, la palabra function, el nombre y los
  ! argumentos ficticios entre paréntesis.
  real(dp) function altura_maxima(v0, theta_grados)
    real(dp), intent(in) :: v0, theta_grados      ! argumentos: solo entran
    real(dp), parameter :: g = 9.8_dp, pi = 4.0_dp*atan(1.0_dp)   ! constantes locales
    real(dp) :: theta                             ! variable local: ángulo en radianes

    theta = theta_grados*pi/180.0_dp              ! sin() recibe radianes
    altura_maxima = (v0*sin(theta))**2/(2.0_dp*g) ! el resultado se asigna al nombre de la función
  end function altura_maxima                      ! aquí vuelve el control al programa principal

end program funcion_altura

In [ ]:
!gfortran -Wall funcion_altura.f90 -o funcion_altura && ./funcion_altura

### 2.2 Python: `def` y `return`

En Python, la función se define con `def`, y `return` indica el valor que devuelve. Una función sin `return` devuelve `None`.

In [ ]:
# Semana 6 -- La misma función en Python
import math

# def: nombre de la función y sus parámetros; el cuerpo va indentado.
def altura_maxima(v0, theta_grados):
    """Altura máxima de un proyectil lanzado desde el suelo.

    Entradas: v0 [m/s], theta_grados [grados]
    Salida:   altura máxima [m]
    """
    g = 9.8                                   # variable local
    theta = math.radians(theta_grados)        # sin() recibe radianes
    return (v0*math.sin(theta))**2/(2*g)      # return: valor que devuelve la función

# --- Tres llamadas a la misma función, con los casos del Taller 1 ---
for v0, th in [(9.8, 90), (20.0, 45), (20.0, 30)]:
    print(f"v0 = {v0:4.1f}, theta = {th}:  h_max = {altura_maxima(v0, th):10.4f}")

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

Los dos lenguajes dan los mismos valores de la tabla de la sección 1. El cálculo está escrito **una sola vez**: si la fórmula tuviera un error, se corrige en un solo lugar y queda corregido para todos los casos.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Si una misma fórmula aparece dos veces en un programa, conviértala en una función. Dele un nombre que diga qué calcula (`altura_maxima`, no `f1`) y un comentario con las unidades de las entradas y de la salida.

</div>

## 3. Subrutinas: varios resultados

Un tiro parabólico tiene tres resultados naturales: altura máxima, alcance y tiempo de vuelo,

$$h_{\max} = \frac{v_0^2\sin^2\theta}{2g}, \qquad R = \frac{v_0^2 \sin 2\theta}{g}, \qquad t_v = \frac{2 v_0 \sin\theta}{g},$$

donde $h_{\max}$ es la altura máxima [m], $R$ el alcance o distancia horizontal recorrida hasta volver a la altura de lanzamiento [m], y $t_v$ el tiempo de vuelo [s]. Las tres dependen de los mismos datos: la rapidez inicial $v_0$, el ángulo $\theta$ y la gravedad $g$. El factor $\sin 2\theta = 2\sin\theta\cos\theta$ combina la componente horizontal de la velocidad, $v_0\cos\theta$, con el tiempo de vuelo.

### 3.1 Fortran: `subroutine` y `call`

En los ejemplos de esta semana, cada función de Fortran devuelve un resultado escalar. Para entregar varios resultados separados se usa una **subrutina**: no devuelve nada por su nombre, sino que escribe sus resultados en argumentos declarados `intent(out)`. Se llama con `call`.

In [ ]:
%%writefile subrutina_tiro.f90
! Semana 6 -- Una subrutina en Fortran: tres resultados del tiro parabólico
program subrutina_tiro
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp) :: h, r, t                               ! recibirán los resultados

  ! --- call: la subrutina se ejecuta como una instrucción aparte ---
  ! Los dos primeros argumentos son datos; h, r y t reciben los resultados.
  call tiro_parabolico(20.0_dp, 45.0_dp, h, r, t)
  print '(a, f10.4, a)', 'Altura máxima:   ', h, ' m'
  print '(a, f10.4, a)', 'Alcance:         ', r, ' m'
  print '(a, f10.4, a)', 'Tiempo de vuelo: ', t, ' s'

contains

  ! ------------------------------------------------------------------
  ! tiro_parabolico: altura máxima, alcance y tiempo de vuelo
  !   Entradas: v0 [m/s], theta_grados [grados]
  !   Salidas:  h_max [m], alcance [m], t_vuelo [s]
  ! ------------------------------------------------------------------
  ! Una subrutina no tiene tipo: su encabezado solo lleva el nombre y los argumentos.
  subroutine tiro_parabolico(v0, theta_grados, h_max, alcance, t_vuelo)
    real(dp), intent(in)  :: v0, theta_grados       ! entran
    real(dp), intent(out) :: h_max, alcance, t_vuelo ! salen: la subrutina les asigna valor
    real(dp), parameter :: g = 9.8_dp, pi = 4.0_dp*atan(1.0_dp)
    real(dp) :: theta                               ! local

    theta   = theta_grados*pi/180.0_dp
    h_max   = (v0*sin(theta))**2/(2.0_dp*g)
    alcance = v0**2*sin(2.0_dp*theta)/g
    t_vuelo = 2.0_dp*v0*sin(theta)/g
  end subroutine tiro_parabolico

end program subrutina_tiro

In [ ]:
!gfortran -Wall subrutina_tiro.f90 -o subrutina_tiro && ./subrutina_tiro

### 3.2 Python: se devuelve una tupla

En Python no hace falta otra construcción: una función puede devolver varios valores separados por comas (una *tupla*), y la llamada los reparte entre varias variables.

In [ ]:
# Semana 6 -- Varios resultados en Python: se devuelve una tupla
import math

def tiro_parabolico(v0, theta_grados):
    """Altura máxima, alcance y tiempo de vuelo de un tiro parabólico.

    Entradas: v0 [m/s], theta_grados [grados]
    Salida:   tupla (h_max [m], alcance [m], t_vuelo [s])
    """
    g = 9.8
    theta = math.radians(theta_grados)
    h_max   = (v0*math.sin(theta))**2/(2*g)
    alcance = v0**2*math.sin(2*theta)/g
    t_vuelo = 2*v0*math.sin(theta)/g
    return h_max, alcance, t_vuelo           # tres valores separados por comas: una tupla

h, r, t = tiro_parabolico(20.0, 45.0)        # la tupla se reparte en tres variables, en orden
print(f"Altura máxima:   {h:10.4f} m")
print(f"Alcance:         {r:10.4f} m")
print(f"Tiempo de vuelo: {t:10.4f} s")

### 3.3 ¿Función o subrutina?

En Fortran hay dos clases de procedimientos, y no son intercambiables:

- Una **función** (`function`) calcula y devuelve **un solo valor** a través de su propio nombre. Por eso se usa *dentro de una expresión*, igual que las funciones intrínsecas `sin(x)` o `sqrt(x)`: `y = 2.0_dp*altura_maxima(v0, theta)`, o `if (altura_maxima(v0, theta) > 10.0_dp) ...`.
- Una **subrutina** (`subroutine`) realiza una **tarea**. No tiene valor propio: se ejecuta como una instrucción aparte, con `call`, y comunica sus resultados a través de los argumentos declarados `intent(out)` o `intent(inout)`. Ya se usaron subrutinas intrínsecas en la Semana 5: `call random_number(u)` entrega su resultado en el argumento `u`, y `call random_seed(put=semilla)` cambia el estado del generador.

| | Función | Subrutina |
|---|---|---|
| Cómo se invoca | Dentro de una expresión: `y = f(x)` | Como instrucción: `call s(x, y, z)` |
| Qué entrega | Un valor, por su nombre | Cero, uno o varios resultados, por sus argumentos |
| Sus argumentos (regla de este curso) | Todos `intent(in)` | `intent(in)`, `intent(out)` o `intent(inout)` |
| Puede escribir archivos o imprimir | No debería | Sí |
| Intrínsecas que ya conoce | `sin`, `sqrt`, `abs`, `mod`, `real` | `random_number`, `random_seed` |
| En esta semana | `altura_maxima`, `alcance` | `tiro_parabolico` (tres resultados), `imprimir_seno` (imprime) |

La restricción de usar solo `intent(in)` en las funciones es una **regla de diseño del curso**, no una limitación del lenguaje Fortran. Aquí buscamos funciones que calculen un resultado sin modificar datos de entrada.

En Python solo existe `def`: una función que devuelve un valor con `return` corresponde a una `function` de Fortran; una que devuelve una tupla o que solo realiza una acción (imprimir, escribir un archivo) cumple el papel de una `subroutine`.

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo: cómo elegir.**

1. Si el procedimiento produce **un solo resultado** y no hace nada más, escriba una **función**.
2. Si produce **varios resultados**, escriba una **subrutina** con un argumento `intent(out)` para cada uno.
3. Si realiza una **acción** (escribir un archivo, imprimir una tabla, modificar datos), escriba una **subrutina**, aunque no devuelva nada.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** las funciones que impriman, escriban archivos o modifiquen sus argumentos: el lector de `y = f(x)` espera que solo se calcule `y`. Como regla de diseño del curso, si un procedimiento necesita hacer algo más que devolver un valor, escríbalo como subrutina. Tampoco intente usar una subrutina dentro de una expresión: `y = 2*tiro_parabolico(...)` no compila.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 1 (no calificado): ¿función o subrutina?**

Escriba un programa `ejercicio_tiempo.f90` con un procedimiento interno que calcule el tiempo de vuelo $t_v = 2v_0\sin\theta/g$. Elija entre función y subrutina con la regla de la sección 3.3 y justifique la elección en una línea de comentario.

**Debe obtener:** $t_v = 2.8862$ s para $v_0 = 20$ m/s y $\theta = 45^\circ$, y $t_v = 2.0000$ s para $v_0 = 9.8$ m/s y $\theta = 90^\circ$.

</div>

In [ ]:
# Ejercicio 1: programa (%%writefile ejercicio_tiempo.f90)

In [ ]:
# Ejercicio 1: compilación y ejecución

## 4. Paso de argumentos

### 4.1 Fortran: asociación de argumentos e `intent`

En una llamada, los argumentos ficticios del procedimiento se asocian con los argumentos reales. En los ejemplos escalares siguientes, modificar el argumento ficticio modifica la variable del programa que llamó al procedimiento. Esto suele explicarse como **paso por referencia**. No significa que siempre se use exactamente la misma posición de memoria: el compilador puede utilizar temporales cuando corresponde.

`intent` declara cómo debe usarse cada argumento; no selecciona el mecanismo de paso de datos.

El programa siguiente convierte un ángulo a radianes *dentro* de la subrutina, sobre el propio argumento. Después de la llamada, la variable `angulo` del programa principal ya no vale 30.


In [ ]:
%%writefile referencia.f90
! Semana 6 -- Paso por referencia: la subrutina modifica la variable del programa
program referencia
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp) :: angulo                                ! ángulo en grados

  angulo = 30.0_dp
  print '(a, f10.6)', 'Antes de la llamada:   angulo =', angulo
  call imprimir_seno(angulo)                        ! modificar theta cambia aquí el valor de angulo
  print '(a, f10.6)', 'Después de la llamada: angulo =', angulo   ! ya no es 30

contains

  ! ------------------------------------------------------------------
  ! imprimir_seno: imprime el seno de un ángulo
  !   Entrada: theta [grados]   (sin intent: EJEMPLO DE LO QUE NO SE DEBE HACER)
  !   Salida:  ninguna; imprime en pantalla
  ! ------------------------------------------------------------------
  subroutine imprimir_seno(theta)
    real(dp) :: theta                               ! sin intent: nada impide modificarlo
    theta = theta*4.0_dp*atan(1.0_dp)/180.0_dp      ! ¡modifica la variable del llamador!
    print '(a, f10.6)', '  sin(theta) =', sin(theta)
  end subroutine imprimir_seno

end program referencia

In [ ]:
!gfortran referencia.f90 -o referencia && ./referencia

Si el programa usara `angulo` después (por ejemplo, para imprimir una tabla en grados), el resultado sería incorrecto, y el error no estaría en la línea donde se ve. La protección es declarar `intent(in)`: el compilador rechaza cualquier asignación al argumento.

In [ ]:
%%writefile referencia_intent.f90
! Semana 6 -- Con intent(in), el compilador detecta la modificación
program referencia_intent
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp) :: angulo                                ! ángulo en grados

  angulo = 30.0_dp
  call imprimir_seno(angulo)

contains

  ! ------------------------------------------------------------------
  ! imprimir_seno: imprime el seno de un ángulo
  !   Entrada: theta [grados], protegida con intent(in)
  !   Salida:  ninguna; imprime en pantalla
  ! ------------------------------------------------------------------
  subroutine imprimir_seno(theta)
    real(dp), intent(in) :: theta                   ! solo entra: no se puede asignar
    theta = theta*4.0_dp*atan(1.0_dp)/180.0_dp      ! esta línea produce el error de compilación
    print '(a, f10.6)', '  sin(theta) =', sin(theta)
  end subroutine imprimir_seno

end program referencia_intent

In [ ]:
# Se espera un error de compilación: es el objetivo del ejemplo
!gfortran -fdiagnostics-color=never referencia_intent.f90 -o referencia_intent

La corrección es usar una **variable local** para el ángulo en radianes, como en la función `altura_maxima` de la sección 2.

| `intent` | Significado | Si se viola |
|---|---|---|
| `in` | El argumento solo entra: no se puede modificar. | Error de compilación. |
| `out` | El argumento solo sale: su valor de entrada no se usa. | El valor de entrada se pierde. |
| `inout` | Entra con un valor y sale modificado. | — |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo (regla del curso).** Declare `intent` en todos los argumentos de todos los procedimientos. Explicita qué entra y qué sale, y permite al compilador detectar modificaciones indebidas de argumentos `intent(in)`. No sustituye la revisión del algoritmo ni garantiza que se asignen correctamente todos los resultados.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** reutilizar un argumento de entrada como variable de trabajo (por ejemplo, convertir `theta` a radianes sobre el mismo argumento). Use una variable local.

</div>

### 4.2 Python: el parámetro recibe el valor

En Python, al llamar a una función, el parámetro pasa a nombrar el mismo objeto que el argumento. Pero los números son **inmutables**: la instrucción `theta = theta*pi/180` no cambia el objeto, sino que crea uno nuevo y hace que el nombre local `theta` apunte a él. La variable del llamador no cambia.

In [ ]:
# Semana 6 -- En Python, reasignar un parámetro no afecta al llamador
import math

def imprimir_seno(theta):
    """Imprime el seno de un ángulo.

    Entrada: theta [grados]
    Salida:  ninguna (imprime en pantalla)
    """
    theta = math.radians(theta)        # crea un objeto nuevo; solo cambia el nombre local
    print(f"  sin(theta) = {math.sin(theta):.6f}")

angulo = 30.0
print(f"Antes de la llamada:   angulo = {angulo}")
imprimir_seno(angulo)
print(f"Después de la llamada: angulo = {angulo}")   # sigue siendo 30.0

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

Con objetos **mutables**, como las listas, una función sí puede modificar los datos del llamador (por ejemplo, con `lista.append(x)`). Se retomará con los arreglos en la Semana 7.

</div>

## 5. Variables locales y globales

Una variable **local** existe solo dentro del procedimiento que la declara. Una variable **global** es visible desde varios procedimientos. Cada lenguaje tiene sus propias reglas, y cada uno tiene un error típico.

### 5.1 Python: leer una global sí; asignarla, no

Dentro de una función, Python decide si un nombre es local mirando si la función **lo asigna** en algún lugar. Si solo lo lee, busca la variable global. Si lo asigna, el nombre es local en toda la función, incluso en las líneas anteriores a la asignación.

In [1]:
# Semana 6 -- Variables globales en Python
G = 9.8                          # global: definida fuera de toda función

def peso(m):
    """Peso [N] de una masa m [kg]. Solo LEE la global G."""
    return m*G                   # no asigna G: Python busca la variable global

print(f"peso(2.0) = {peso(2.0)} N")

contador = 0                     # global que se quiere modificar

def contar_llamada():
    """Intenta incrementar la global contador (EJEMPLO DE ERROR)."""
    contador = contador + 1      # asigna: contador es LOCAL en toda la función
    return contador

# --- try/except captura el error para mostrarlo sin detener el notebook ---
try:
    contar_llamada()
except UnboundLocalError as error:
    print("UnboundLocalError:", error)

peso(2.0) = 19.6 N
UnboundLocalError: local variable 'contador' referenced before assignment


El error dice que la variable local `contador` se lee antes de tener valor: el lado derecho de `contador = contador + 1` ya se refiere a la local, que todavía no existe. La instrucción `global` le indica a Python que el nombre se refiere a la variable global:

In [2]:
# Con la instrucción global, la función modifica la variable global
contador = 0

def contar_llamada():
    global contador
    contador = contador + 1
    return contador

for _ in range(3):
    print("llamada número", contar_llamada())

llamada número 1
llamada número 2
llamada número 3


<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo (regla del curso).** Use variables globales solo para **constantes**: $g$, $\pi$, los coeficientes de un modelo. Los resultados se devuelven con `return`.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** guardar resultados en variables globales o modificarlas dentro de una función: una función que cambia una global puede alterar el resultado de otra parte del programa sin que se vea en la llamada.

</div>

### 5.2 Fortran: el procedimiento interno ve las variables del programa

Un procedimiento escrito después de `contains` ve todas las variables del programa que lo contiene (*asociación por anfitrión*). Si el procedimiento usa un nombre que no declaró, Fortran usa el del programa principal, sin ningún aviso.

En el programa siguiente, la función `factorial` usa `k` como contador de su ciclo, pero no lo declara: usa el `k` del programa principal, que cuenta los casos. Se esperaban tres líneas.

In [ ]:
%%writefile anfitrion.f90
! Semana 6 -- Error típico: un procedimiento interno usa una variable del programa
program anfitrion
  implicit none
  integer :: k, f                  ! k: número de caso; f: resultado del factorial

  ! --- Se esperan tres casos: k = 1, 2, 3 ---
  k = 1
  do while (k <= 3)
     f = factorial(4)
     print '(a, i2, a, i4)', 'caso', k, ':  4! =', f
     k = k + 1
  end do

contains

  ! ------------------------------------------------------------------
  ! factorial: n! = 1*2*...*n
  !   Entrada: n (entero >= 0)
  !   Salida:  n!, como valor de la función
  ! ------------------------------------------------------------------
  integer function factorial(n)
    integer, intent(in) :: n
    factorial = 1
    do k = 2, n                  ! k NO está declarada aquí: es la del programa principal
       factorial = factorial*k
    end do
  end function factorial

end program anfitrion

In [ ]:
!gfortran -Wall anfitrion.f90 -o anfitrion && ./anfitrion

Aparece **una sola** línea, y dice «caso 5»: al terminar su ciclo, `factorial` dejó `k = 5`, y el ciclo del programa principal terminó después de la primera vuelta. El programa compila sin advertencias. La corrección es declarar `integer :: k` dentro de la función, para que sea una variable local.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** usar dentro de un procedimiento una variable que no declaró en él. Con `implicit none` el compilador detecta las variables no declaradas en un programa, pero **no** las de un procedimiento interno, porque las encuentra en el programa anfitrión.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Declare dentro de cada procedimiento todas sus variables de trabajo (contadores, auxiliares). Así cada procedimiento depende solo de sus argumentos y de las constantes de los módulos.

</div>

### 5.3 Fortran: una inicialización en la declaración se conserva entre llamadas

En Fortran, una variable local que recibe valor **en su declaración** (`integer :: n = 0`) adquiere el atributo `save`: se inicializa una sola vez, al empezar el programa, y conserva su valor entre una llamada y la siguiente. Una asignación en el cuerpo (`n = 0`) se ejecuta en cada llamada.

In [ ]:
%%writefile save_implicito.f90
! Semana 6 -- Inicialización en la declaración = atributo save implícito
program save_implicito
  implicit none
  integer :: k

  ! --- Tres llamadas a cada subrutina ---
  do k = 1, 3
     call con_save()
     call sin_save()
  end do

contains

  ! ------------------------------------------------------------------
  ! con_save: cuenta sus llamadas con una variable inicializada en la declaración
  !   Entradas: ninguna;  Salida: ninguna (imprime el número de llamada)
  ! ------------------------------------------------------------------
  subroutine con_save()
    integer :: llamadas = 0       ! se inicializa UNA vez: conserva su valor (save implícito)
    llamadas = llamadas + 1
    print '(a, i2)', 'con valor en la declaración: llamada', llamadas
  end subroutine con_save

  ! ------------------------------------------------------------------
  ! sin_save: la misma cuenta, con la variable inicializada en el cuerpo
  !   Entradas: ninguna;  Salida: ninguna (imprime el número de llamada)
  ! ------------------------------------------------------------------
  subroutine sin_save()
    integer :: llamadas
    llamadas = 0                  ! asignación: se ejecuta en CADA llamada
    llamadas = llamadas + 1
    print '(a, i2)', 'con asignación en el cuerpo: llamada', llamadas
  end subroutine sin_save

end program save_implicito

In [ ]:
!gfortran save_implicito.f90 -o save_implicito && ./save_implicito

En Python esto no ocurre: las variables locales se crean de nuevo en cada llamada. Por eso, en la sección 5.1, el contador tuvo que ser global.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** dar valor en la declaración (`real(dp) :: suma = 0.0_dp`) a una variable local que debe empezar desde cero en cada llamada, como un acumulador: desde la segunda llamada empezará con el valor que dejó la anterior.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Inicialice los acumuladores con una asignación en el cuerpo del procedimiento. Si de verdad necesita que una variable conserve su valor entre llamadas, declárelo explícitamente con el atributo `save`, para que se vea en el código.

</div>

| | Fortran | Python |
|---|---|---|
| Variable local | Declarada dentro del procedimiento. | Asignada dentro de la función. |
| Global | Variables de un `module` (sección 6) o del programa anfitrión. | Asignadas fuera de toda función. |
| Error típico | Usar sin declarar una variable del anfitrión (5.2). | Asignar una global sin `global` (5.1). |
| Valor entre llamadas (variables locales de estos ejemplos) | Se conserva si tiene `save`, incluido el implícito de una inicialización en la declaración (5.3). | Una asignación local se ejecuta de nuevo en cada llamada. |

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 2 (no calificado): errores de ámbito.**

El programa siguiente debería imprimir, para $n = 1, 2, 3$, el promedio de los cuadrados $\frac{1}{n}\sum_{j=1}^{n} j^2$. Compila sin advertencias, pero tiene **dos** errores de los que se vieron en las secciones 5.2 y 5.3. Ejecútelo, identifique los dos errores y escriba una versión corregida en `ambito_corregido.f90`, con un comentario en cada corrección.

**Debe obtener** tres líneas: $n = 1$: 1.0000; $n = 2$: 2.5000; $n = 3$: 4.6667.

</div>

In [ ]:
%%writefile ambito_con_errores.f90
! Ejercicio 2 -- Programa con dos errores de ámbito (NO MODIFICAR: copie y corrija)
program ambito_con_errores
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer :: j, n_terminos
  real(dp) :: p

  j = 1
  do while (j <= 3)
     n_terminos = j
     p = promedio_cuadrados(n_terminos)
     print '(a, i2, a, f10.4)', 'n =', n_terminos, ':  promedio de j^2 =', p
     j = j + 1
  end do

contains

  ! ------------------------------------------------------------------
  ! promedio_cuadrados: (1/n) * suma de j^2 para j = 1, ..., n
  !   Entrada: n (número de términos)
  !   Salida:  el promedio, como valor de la función
  ! ------------------------------------------------------------------
  real(dp) function promedio_cuadrados(n)
    integer, intent(in) :: n
    real(dp) :: suma = 0.0_dp
    do j = 1, n
       suma = suma + real(j, dp)**2
    end do
    promedio_cuadrados = suma/real(n, dp)
  end function promedio_cuadrados

end program ambito_con_errores

In [ ]:
!gfortran -Wall ambito_con_errores.f90 -o ambito_con_errores && ./ambito_con_errores

In [ ]:
# Ejercicio 2: programa corregido (%%writefile ambito_corregido.f90)

In [ ]:
# Ejercicio 2: compilación y ejecución

## 6. Módulos

Un **módulo** agrupa constantes y procedimientos en un archivo propio, para que varios programas los usen sin copiarlos. Es el lugar natural de las constantes globales de un problema.

### 6.1 Fortran: `module`, `use` y el orden de compilación

El módulo declara sus constantes al principio y sus procedimientos después de `contains`. Un programa accede a ellos con `use nombre_del_modulo`, que debe ir antes de `implicit none`. Con `use ..., only: lista` se importa solo lo que se nombra.

In [ ]:
%%writefile mod_proyectil.f90
! Semana 6 -- Módulo con las constantes y los procedimientos del proyectil
! Un módulo no se ejecuta solo: lo usan otros programas con la instrucción use.
module mod_proyectil
  implicit none

  ! --- Parte de declaraciones: constantes globales del módulo ---
  ! Todo programa que haga "use mod_proyectil" puede usarlas.
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: g  = 9.8_dp                 ! gravedad [m/s^2]
  real(dp), parameter :: pi = 4.0_dp*atan(1.0_dp)

! --- Parte de procedimientos: después de contains ---
contains

  ! ------------------------------------------------------------------
  ! altura_maxima: altura máxima de un proyectil lanzado desde el suelo
  !   Entradas: v0 [m/s], theta_grados [grados]
  !   Salida:   altura máxima [m]
  ! ------------------------------------------------------------------
  real(dp) function altura_maxima(v0, theta_grados)
    real(dp), intent(in) :: v0, theta_grados
    altura_maxima = (v0*sin(theta_grados*pi/180.0_dp))**2/(2.0_dp*g)   ! usa g y pi del módulo
  end function altura_maxima

  ! ------------------------------------------------------------------
  ! alcance: distancia horizontal hasta volver a la altura de lanzamiento
  !   Entradas: v0 [m/s], theta_grados [grados]
  !   Salida:   alcance [m]
  ! ------------------------------------------------------------------
  real(dp) function alcance(v0, theta_grados)
    real(dp), intent(in) :: v0, theta_grados
    alcance = v0**2*sin(2.0_dp*theta_grados*pi/180.0_dp)/g
  end function alcance

end module mod_proyectil

In [ ]:
%%writefile usa_modulo.f90
! Semana 6 -- Programa que usa el módulo mod_proyectil
program usa_modulo
  ! use va antes de implicit none. Con only: se importan solo los nombres listados;
  ! g y pi quedan dentro del módulo, porque este programa no los necesita.
  use mod_proyectil, only: dp, altura_maxima, alcance
  implicit none

  print '(a, f10.4, a)', 'h_max = ', altura_maxima(20.0_dp, 45.0_dp), ' m'
  print '(a, f10.4, a)', 'R     = ', alcance(20.0_dp, 45.0_dp), ' m'
end program usa_modulo

Al compilar el módulo, `gfortran` genera el archivo `mod_proyectil.mod`, que describe sus constantes y procedimientos. El programa que hace `use mod_proyectil` necesita ese archivo, así que **el módulo se compila primero**. En el orden inverso, la compilación falla:

In [ ]:
# Orden incorrecto: el programa antes que el módulo (se espera un error)
!rm -f mod_proyectil.mod
!gfortran -fdiagnostics-color=never usa_modulo.f90 mod_proyectil.f90 -o usa_modulo

In [ ]:
# Orden correcto: primero el módulo, después el programa
!gfortran -Wall mod_proyectil.f90 usa_modulo.f90 -o usa_modulo && ./usa_modulo

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Ponga cada módulo en su propio archivo, con el mismo nombre del módulo, y compílelo siempre primero. Use `use ..., only:` para que se vea de dónde viene cada nombre.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** escribir `use` después de `implicit none`: el compilador lo rechaza. El orden dentro de un programa es `program`, `use`, `implicit none`, declaraciones e instrucciones.

</div>

### 6.2 Python: un archivo `.py` es un módulo

En Python, cualquier archivo `.py` es un módulo. Se escribe con `%%writefile` y se importa con `import`. Las constantes del módulo se usan con el prefijo del módulo (`proyectil.G`) o se importan por nombre con `from ... import ...`.

In [3]:
%%writefile proyectil.py
"""Semana 6 -- Módulo de Python con las constantes y funciones del proyectil.

Se usa con:  import proyectil   o   from proyectil import altura_maxima
"""
import math

G = 9.8                          # constante global del módulo: gravedad [m/s^2]

def altura_maxima(v0, theta_grados):
    """Altura máxima de un proyectil lanzado desde el suelo.

    Entradas: v0 [m/s], theta_grados [grados]
    Salida:   altura máxima [m]
    """
    return (v0*math.sin(math.radians(theta_grados)))**2/(2*G)

def alcance(v0, theta_grados):
    """Distancia horizontal hasta volver a la altura de lanzamiento.

    Entradas: v0 [m/s], theta_grados [grados]
    Salida:   alcance [m]
    """
    return v0**2*math.sin(2*math.radians(theta_grados))/G

Overwriting proyectil.py


In [4]:
# Uso del módulo proyectil.py
import importlib
import proyectil
importlib.reload(proyectil)      # vuelve a leer el archivo si se modificó

print(f"h_max = {proyectil.altura_maxima(20.0, 45.0):10.4f} m")
print(f"R     = {proyectil.alcance(20.0, 45.0):10.4f} m")
print(f"g     = {proyectil.G} m/s^2")

h_max =    10.2041 m
R     =    40.8163 m
g     = 9.8 m/s^2


<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo: `importlib.reload`.** Python importa un módulo una sola vez por sesión. Si se modifica `proyectil.py` y se vuelve a ejecutar `import proyectil`, se sigue usando la versión anterior. `importlib.reload` obliga a leer el archivo de nuevo.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 3 (no calificado): el Taller en pequeño.**

Lleve el procedimiento del Ejercicio 1 a un módulo, en los dos lenguajes:

1. Un módulo de Fortran `mod_tiro`, en el archivo `mod_tiro.f90`, con las constantes $g$ y $\pi$ y el procedimiento del tiempo de vuelo.
2. Un programa `usa_tiro.f90` que lo use con `use ..., only:` e imprima $t_v$ para $v_0 = 20$ m/s y $\theta = 45^\circ$. Compile en el orden correcto.
3. Un módulo de Python `tiro.py` con la misma función, importado desde una celda que imprima el mismo caso.

**Debe obtener** $t_v = 2.8862$ s en Fortran y en Python.

</div>

In [ ]:
# Ejercicio 3: módulo de Fortran (%%writefile mod_tiro.f90)

In [ ]:
# Ejercicio 3: programa que usa el módulo (%%writefile usa_tiro.f90)

In [ ]:
# Ejercicio 3: compilación (primero el módulo) y ejecución

In [ ]:
# Ejercicio 3: módulo de Python (%%writefile tiro.py)

In [ ]:
# Ejercicio 3: uso del módulo de Python

## 7. Ejemplo integrador: el proyectil del Taller 1 en dos lenguajes

Se reúne todo lo anterior en el flujo de trabajo del Taller 6: **Fortran calcula y escribe un archivo; Python lo lee, lo grafica y comprueba el resultado con su propia implementación.**

El programa de Fortran usa el módulo `mod_proyectil` de la sección 6.1 y agrega:

- una función lógica `verifica` que compara un valor con su referencia, con tolerancia relativa;
- los tres casos del Taller 1, que ahora son tres llamadas y no tres ejecuciones;
- un archivo `tabla_proyectil.dat` con $h_{\max}$ y $R$ para $\theta = 5^\circ, 10^\circ, \dots, 85^\circ$ con $v_0 = 20$ m/s, escrito con un ciclo.

In [ ]:
%%writefile proyectil_completo.f90
! Semana 6 -- Ejemplo integrador: verificación y tabla del proyectil con un módulo
! Salida: tabla_proyectil.dat  (theta [grados], h_max [m], alcance [m])
program proyectil_completo
  use mod_proyectil, only: dp, altura_maxima, alcance    ! procedimientos del módulo
  implicit none
  integer :: k, u_dat               ! k: contador; u_dat: unidad del archivo de salida
  real(dp) :: theta                 ! ángulo [grados]

  ! --- Verificación: los tres casos del Taller 1, ahora tres llamadas ---
  print '(a)', 'Verificación (casos del Taller 1):'
  call reporta( 9.8_dp, 90.0_dp,  4.9000_dp)
  call reporta(20.0_dp, 45.0_dp, 10.2041_dp)
  call reporta(20.0_dp, 30.0_dp,  5.1020_dp)

  ! --- Tabla para graficar en Python: theta = 5, 10, ..., 85 grados ---
  ! newunit= elige un número de unidad libre y lo guarda en u_dat.
  open(newunit=u_dat, file='tabla_proyectil.dat', status='replace')
  do k = 1, 17
     theta = 5.0_dp*k
     write(u_dat, '(f6.1, 2f14.8)') theta, altura_maxima(20.0_dp, theta), alcance(20.0_dp, theta)
  end do
  close(u_dat)
  print '(a)', 'Archivo escrito: tabla_proyectil.dat'

contains

  ! ------------------------------------------------------------------
  ! verifica: compara un valor con su referencia, con tolerancia relativa
  !   Entradas: valor, referencia, tol (tolerancia relativa)
  !   Salida:   .true. si |valor - referencia| < tol*|referencia|
  ! ------------------------------------------------------------------
  ! Una función puede devolver cualquier tipo: aquí, un valor lógico.
  logical function verifica(valor, referencia, tol)
    real(dp), intent(in) :: valor, referencia, tol
    verifica = abs(valor - referencia) < tol*abs(referencia)
  end function verifica

  ! ------------------------------------------------------------------
  ! reporta: calcula h_max e imprime una línea con PASS o FAIL
  !   Entradas: v0 [m/s], theta_grados [grados], h_ref [m] (referencia)
  !   Salida:   ninguna; imprime en pantalla (por eso es subrutina, sección 3.3)
  ! ------------------------------------------------------------------
  subroutine reporta(v0, theta_grados, h_ref)
    real(dp), intent(in) :: v0, theta_grados, h_ref
    real(dp) :: h                                   ! local: altura calculada
    character(len=4) :: veredicto                   ! local: 'PASS' o 'FAIL'

    h = altura_maxima(v0, theta_grados)             ! una subrutina puede llamar a una función
    veredicto = 'FAIL'
    if (verifica(h, h_ref, 1.0e-4_dp)) veredicto = 'PASS'   ! referencias con 4 decimales
    print '(a, f5.1, a, f5.1, a, f10.4, a, f10.4, 2x, a)', '  v0 =', v0, '  theta =', theta_grados, &
          '  h =', h, '  ref =', h_ref, veredicto
  end subroutine reporta

end program proyectil_completo

In [ ]:
!gfortran -Wall mod_proyectil.f90 proyectil_completo.f90 -o proyectil_completo && ./proyectil_completo

La celda de Python lee la tabla, la compara con el módulo `proyectil.py` de la sección 6.2 y la grafica. Si los dos lenguajes implementan el mismo modelo, la diferencia máxima debe ser del orden del redondeo de los 8 decimales del archivo.

In [ ]:
# Python: lectura de la tabla de Fortran, comparación con proyectil.py y gráfica
import numpy as np
import matplotlib.pyplot as plt
import proyectil

AZUL, DORADO, ROJO = "#162A54", "#C9972C", "#B02A20"     # colores del curso

def leer_tabla(archivo):
    """Lee la tabla escrita por proyectil_completo.f90.

    Entrada: nombre del archivo
    Salida:  tupla (theta [grados], h_max [m], alcance [m]), cada una una columna
    """
    datos = np.loadtxt(archivo)
    return datos[:, 0], datos[:, 1], datos[:, 2]

def diferencia_maxima(theta, h_fortran, r_fortran, v0):
    """Mayor diferencia absoluta entre la tabla de Fortran y el módulo de Python.

    Entradas: columnas leídas con leer_tabla y la rapidez v0 [m/s] usada en Fortran
    Salida:   diferencia máxima [m]
    """
    dif = 0.0                                          # acumulador del máximo
    for th, h, r in zip(theta, h_fortran, r_fortran):  # recorre las filas de la tabla
        dif = max(dif, abs(h - proyectil.altura_maxima(v0, th)),
                       abs(r - proyectil.alcance(v0, th)))
    return dif

# --- Comparación entre lenguajes ---
theta, h, r = leer_tabla("tabla_proyectil.dat")
print(f"Diferencia máxima Fortran - Python: {diferencia_maxima(theta, h, r, 20.0):.1e} m")

# --- Gráfica ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(theta, r, "o-", color=AZUL, label="alcance $R$")
ax.plot(theta, h, "s-", color=DORADO, label="altura máxima $h_{\\max}$")
ax.axvline(45, color=ROJO, ls="--", lw=1)              # máximo alcance en 45°
ax.set_xlabel(r"$\theta$ [grados]"); ax.set_ylabel("distancia [m]")
ax.set_title(r"Proyectil con $v_0 = 20$ m/s (calculado en Fortran)")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Recapitulación: lo que reúne el Taller 6

El Taller 6 cierra el Capítulo 2 (Estructura del programa). Además de los procedimientos y módulos de esta semana, usa casi todo lo visto desde la Semana 1:

| Lo que pide el Taller | Dónde se vio |
|---|---|
| Completar el pseudocódigo del programa antes de escribir el código | Semana 1 (Taller 1) |
| Trabajar en `real(dp)`, convertir enteros a reales y evitar la división entera en potencias como $A^{-1/3}$ | Semana 2: números pequeños y grandes, precisión |
| Ciclos anidados (para cada $A$, recorrer todos los $Z$) y un condicional que guarda el mínimo encontrado | Semanas 2 a 5: ciclos, condicionales y acumuladores ($\mathcal{C}^3$) |
| Comparar un modelo con datos experimentales y cuantificar la diferencia | Semana 2 (error) y Semana 3 (ajustes: modelo frente a datos) |
| Fortran escribe archivos `.dat`; Python los lee, grafica y analiza | Semanas 4 y 5 |
| Verificar el programa con casos de referencia conocidos (PASS/FAIL), sin sustituir la revisión del código | Todas las semanas |
| Funciones, subrutinas, `intent`, variables locales y globales, módulos | Semana 6, secciones 2 a 6 |

Con procedimientos y módulos, cada cálculo tiene un nombre, recibe sus datos por argumentos y se prueba por separado. El Taller pide resolver con esa organización un problema que no se ha visto en clase.

## 9. Taller 6: Fórmula semiempírica de masa atómica

### A. Enunciado y requisitos

**Contexto físico.** La masa de un núcleo con $Z$ protones y $A$ nucleones es menor que la suma de las masas de sus componentes: la diferencia es la energía de enlace. En el modelo de la gota líquida (Bethe y Weizsäcker), la energía de enlace **por nucleón** es la suma de cuatro términos:

$$E_B(Z,A) = \underbrace{-\,a_V}_{\text{volumen}} \; + \; \underbrace{a_S\,A^{-1/3}}_{\text{superficie}} \; + \; \underbrace{a_C\,Z^2A^{-4/3}}_{\text{Coulomb}} \; + \; \underbrace{a_A\,\frac{(Z - A/2)^2}{A^2}}_{\text{asimetría}}$$

donde $Z$ es el número de protones, $A$ el número de nucleones (protones más neutrones) y $N = A - Z$ el número de neutrones. Cada término tiene un origen físico:

| Término | Expresión | Origen físico | Efecto |
|---|---|---|---|
| Volumen | $-a_V$ | La fuerza nuclear fuerte es de corto alcance: cada nucleón solo interactúa con sus vecinos inmediatos, así que el enlace es proporcional al número de nucleones y **por nucleón es constante**. | Liga ($<0$) |
| Superficie | $a_S\,A^{-1/3}$ | Los nucleones de la superficie tienen menos vecinos. El núcleo es una esfera de radio $R \propto A^{1/3}$; su superficie es $\propto A^{2/3}$ y, dividida entre $A$, da $A^{-1/3}$. | Resta enlace; pesa más en núcleos livianos |
| Coulomb | $a_C\,Z^2A^{-4/3}$ | Repulsión electrostática entre los $Z$ protones, repartidos en la esfera de radio $R \propto A^{1/3}$: la energía total es $\propto Z^2/A^{1/3}$ y, dividida entre $A$, da $Z^2A^{-4/3}$. | Resta enlace; pesa más en núcleos pesados |
| Asimetría | $a_A\,(Z - A/2)^2/A^2$ | Por el principio de exclusión de Pauli, protones y neutrones ocupan niveles de energía separados: este término por sí solo favorece $Z = N = A/2$, y cualquier exceso de uno de los dos cuesta energía. El mínimo del modelo completo depende también de los otros términos. | Resta enlace si $Z \neq A/2$ |

y la masa atómica aproximada se calcula mediante

$$M(Z,A) = Z\,M_H + (A - Z)\,M_n + A\,E_B(Z,A),$$

donde $Z\,M_H$ es la masa de $Z$ átomos de hidrógeno (cada uno, un protón y un electrón), $(A - Z)\,M_n$ la masa de los $N$ neutrones libres, y $A\,E_B$ la energía de enlace **total**, que es negativa y reduce la masa respecto de los componentes separados. Como se utiliza $M_H$, el resultado incluye la masa de los $Z$ electrones: corresponde aproximadamente a una **masa atómica**, despreciando sus energías de enlace electrónicas.

En las ecuaciones se adopta $c=1$: los símbolos de masa representan su energía equivalente $Mc^2$, expresada en **MeV**. Para expresar la masa como tal, la unidad correspondiente es MeV/$c^2$.

Con esta convención, $E_B < 0$ para un núcleo ligado: cuanto más negativo, más ligado. Las tablas experimentales suelen dar la energía de enlace por nucleón con signo positivo, $B/A = -E_B$.

**Constantes** (O. Zubairi y F. Weber, *Introduction to Computational Physics for Undergraduates*, 2.ª ed., IOP, 2024, problema 7.2):

| $a_V$ | $a_S$ | $a_C$ | $a_A$ | $M_n$ | $M_H$ |
|---|---|---|---|---|---|
| 15.8 MeV | 18.3 MeV | 0.71 MeV | 92.7 MeV | 939.57 MeV | 938.78 MeV |
| coeficiente de volumen | coeficiente de superficie | coeficiente de Coulomb | coeficiente de asimetría | masa del neutrón | masa del átomo de hidrógeno |

Los cuatro coeficientes no se deducen de una teoría: se ajustan a las masas medidas de muchos núcleos. Por eso el modelo se llama *semiempírico*.

**Preguntas que debe responder el programa.**

1. ¿Cuánto vale cada término, la energía de enlace por nucleón y la masa atómica aproximada de los núclidos ${}^{244}$Pu, ${}^{70}$Ge, ${}^{179}$Hf, ${}^{38}$Ar, ${}^{56}$Fe, ${}^{40}$Ca, ${}^{235}$U y ${}^{238}$U?
2. Para cada número de nucleones $A$, ¿qué número de protones $Z$ da el núcleo más ligado? El conjunto de estos puntos define la **curva de máximo enlace del modelo**, relacionada con el valle de estabilidad.
3. ¿Cuál es el núcleo con mayor enlace **por nucleón** en toda esa curva, para el intervalo de $A$ estudiado?

Para un $A$ fijo, se busca el núcleo de menor $E_B(Z,A)$ entre $Z = 1, 2, \dots, A$. En todo el taller, «más ligado» se refiere a mayor enlace **por nucleón**, es decir, a un $E_B$ más negativo.

**Alcance físico de la búsqueda.** Minimizar $E_B$ a $A$ fijo no equivale exactamente a minimizar $M$, porque $M_H$ y $M_n$ son diferentes. La estabilidad frente a transformaciones beta requiere comparar las masas de los isóbaros y considerar los canales de decaimiento. En este taller se conserva únicamente la búsqueda del máximo enlace: **no debe cambiarse el criterio de búsqueda por una minimización de la masa**. Para mantener los nombres de archivo y la arquitectura común, se conservan `valle.dat` y el procedimiento `Z_estable`, entendido aquí como el $Z$ de máximo enlace según el modelo.

**Diseño previo (obligatorio)**

En la sección B.1 se entrega un **pseudocódigo marco**: muestra qué procedimientos tiene el programa, cómo se conectan y en qué orden se ejecuta. Antes de escribir código, complete sus tres espacios marcados, como en el Taller 1:

- **[COMPLETE 1]:** cómo se encuentra, para un $A$ dado, el $Z$ de menor $E_B$.
- **[JUSTIFIQUE]:** por qué corresponde usar una subrutina para entregar los dos resultados separados, de acuerdo con la regla de diseño de la sección 3.3.
- **[COMPLETE 2]:** cómo se identifica, durante el ciclo sobre $A$, el núcleo más ligado de toda la curva.

El pseudocódigo completado es un entregable. Escríbalo en español estructurado, no en Fortran. Si su programa final se aparta de él, explique la diferencia debajo del pseudocódigo.

**Requisitos del programa de Fortran**

| | Requisito |
|---|---|
| R1 | Todos los archivos del Taller (código y resultados) van en la carpeta `taller_nucleo_atomico/`. |
| R2 | Las constantes y los procedimientos del modelo están en un **módulo**, en el archivo `modulo_nucleo.f90`. El programa principal está en otro archivo, `programa_nucleo.f90`. |
| R3 | Las constantes son `parameter`. No hay variables globales que no sean constantes. |
| R4 | Cada uno de los cuatro términos es una función. La energía de enlace por nucleón es otra función, que usa las cuatro anteriores; la masa, otra más. |
| R5 | Un procedimiento recibe $A$ y entrega dos resultados: el $Z$ de máximo enlace y su $E_B$. Justifique el uso de una subrutina según la regla de la sección 3.3. |
| R6 | Todos los argumentos de todos los procedimientos tienen `intent`. |
| R7 | Todos los reales son `real(dp)`, con `dp = kind(1.0d0)`. |

**Archivos que debe escribir el programa de Fortran** (columnas separadas por espacios, energías en MeV con al menos 8 decimales, masa con al menos 4):

| Archivo | Filas | Columnas |
|---|---|---|
| `taller_nucleo_atomico/nucleos.dat` | 8, en el orden de la pregunta 1 | $Z$, $A$, $E_{\rm vol}$, $E_{\rm sup}$, $E_{\rm Coul}$, $E_{\rm asim}$, $E_B$, $M$ |
| `taller_nucleo_atomico/valle.dat` | una por cada $A = 4, 5, \dots, 260$ | $A$, $Z$ de máximo enlace, $E_B$ de ese núcleo |
| `taller_nucleo_atomico/maximo.dat` | 1 | $A$, $Z$ y $E_B$ del núcleo más ligado de `valle.dat` |

**Requisitos de la parte de Python**

| | Requisito |
|---|---|
| P1 | Las funciones de lectura y análisis están en el módulo `taller_nucleo_atomico/analisis_nucleo.py`, que se importa desde el notebook. |
| P2 | Una gráfica de $B/A = -E_B$ contra $A$ a lo largo de la curva de máximo enlace, con título, ejes con unidades, leyenda y los valores experimentales de la sección C, guardada en `taller_nucleo_atomico/curva_enlace.png`. |
| P3 | El núcleo más ligado, encontrado en Python a partir de `valle.dat`, escrito en `taller_nucleo_atomico/maximo_python.dat` con el mismo formato de `maximo.dat`. |

**Actividades**

1. Complete el pseudocódigo marco (sección B.1).
2. Escriba, compile y ejecute el programa de Fortran (secciones B.2 y B.3: dos archivos).
3. Escriba el módulo de Python y úselo para la gráfica y el máximo (secciones C.1 a C.4: un módulo y una celda de uso).
4. Ejecute la celda de verificación (sección D). Esta celda no debe modificarse. La entrega debe dar **4/4 PASS**. Esto es necesario, pero no suficiente: también se revisarán el diseño, el código, la gráfica, la interpretación y la sustentación.
5. Responda las preguntas de interpretación (sección E).
6. Elabore el reporte de una página (sección F).

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

No se entrega un esqueleto del programa: la organización en procedimientos es parte de lo que se evalúa. Las secciones 2 a 6 contienen todas las construcciones necesarias. En la sustentación, cada estudiante explicará su propio código.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Construya el programa por partes. Escriba primero el módulo con las funciones de los cuatro términos y pruébelas con el ${}^{244}$Pu, cuyo resultado está en la tabla de la sección D. Solo cuando ese caso coincida, escriba el procedimiento del $Z$ de máximo enlace y el ciclo sobre $A$.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite la división entera.** En Fortran, `-1/3` es una división entre enteros y vale `0`; entonces `a**(-1/3)` vale `1` para cualquier `a`, sin ningún aviso. Escriba los exponentes como reales (`-1.0_dp/3.0_dp`) y convierta los enteros con `real(a, dp)` antes de elevarlos. El mismo cuidado aplica a `A/2` en el término de asimetría cuando $A$ es impar.

</div>

### B. Programa de Fortran

#### B.1 Diseño: pseudocódigo marco

La celda siguiente contiene el marco del programa. Edítela (doble clic) y reemplace cada espacio entre corchetes por su pseudocódigo. No cambie el resto del marco.

```
MÓDULO del modelo
    constantes: a_V, a_S, a_C, a_A, M_n, M_H
    funciones de los cuatro términos               (entradas: Z y/o A; salida: el término [MeV])
    función E_B(Z, A)                              (usa las cuatro funciones anteriores)
    función M(Z, A)                                (usa E_B)
    procedimiento Z_estable                        (entrada: A; resultados: el Z de máximo enlace y su E_B)
        [COMPLETE 1: cómo se encuentra el Z de menor E_B]
        [JUSTIFIQUE: por qué se usa una subrutina para entregar los dos resultados separados]

PROGRAMA PRINCIPAL (usa el módulo)
    para cada uno de los 8 núcleos, en el orden del enunciado:
        escribir una fila de nucleos.dat: Z, A, los cuatro términos, E_B y M
    para A = 4, 5, ..., 260:
        obtener el Z de máximo enlace y su E_B con Z_estable
        escribir una fila de valle.dat: A, Z, E_B
        [COMPLETE 2: cómo se identifica, durante este ciclo, el núcleo más ligado]
    escribir el núcleo más ligado en maximo.dat
```

#### B.2 Qué debe preparar

La parte B consiste en **dos archivos de Fortran**, que se escriben en las dos celdas de la sección B.3, que ya empiezan con la línea `%%writefile` y el nombre del archivo. La tercera celda, ya escrita, los compila y ejecuta; no la modifique.

| Archivo | Qué contiene | Requisitos |
|---|---|---|
| `taller_nucleo_atomico/modulo_nucleo.f90` | El **módulo** del modelo: <br>• las seis constantes de la tabla del enunciado; <br>• una función para cada uno de los cuatro términos; <br>• una función para $E_B(Z,A)$, que use las cuatro anteriores; <br>• una función para $M(Z,A)$; <br>• el procedimiento que, dado $A$, entrega el $Z$ de máximo enlace y su $E_B$. | R2 a R7 |
| `taller_nucleo_atomico/programa_nucleo.f90` | El **programa principal**: usa el módulo y escribe los tres archivos de resultados. Si necesita procedimientos propios (por ejemplo, para escribir una fila), van después de su `contains`. | R2, R6, R7 |

**Resultados de la parte B**, que escribe el programa al ejecutarse, en la misma carpeta y con el formato de la tabla de la sección A: `nucleos.dat`, `valle.dat` y `maximo.dat`.

**La parte B está terminada cuando** la celda de compilación se ejecuta sin errores, los tres archivos existen y la primera fila de `nucleos.dat` (el ${}^{244}$Pu) tiene $E_B \approx -7.5361$ MeV. Esto es un control inicial; la validación completa se realiza en D.

#### B.3 Código de Fortran

In [ ]:
%%writefile taller_nucleo_atomico/modulo_nucleo.f90
! Taller 6 -- Módulo de la fórmula semiempírica (escriba aquí su módulo)

In [ ]:
%%writefile taller_nucleo_atomico/programa_nucleo.f90
! Taller 6 -- Programa principal (escriba aquí su programa)

In [ ]:
# Compilación y ejecución (NO MODIFICAR): primero el módulo, después el programa.
# Se ejecuta dentro de taller_nucleo_atomico/ para que los .dat queden allí.
!cd taller_nucleo_atomico && gfortran -Wall -Wextra modulo_nucleo.f90 programa_nucleo.f90 -o nucleo && ./nucleo

### C. Análisis en Python

#### C.1 Qué debe preparar

La parte C consiste en **un módulo de Python** y **una celda que lo usa**. Trabaja solo con los archivos que escribió la parte B: no vuelve a calcular la fórmula.

| Dónde | Qué contiene | Requisitos |
|---|---|---|
| `taller_nucleo_atomico/analisis_nucleo.py` (celda C.3) | Cuatro funciones, cada una con su docstring de entradas y salidas: <br>• una que lea `valle.dat` y devuelva sus tres columnas; <br>• una que encuentre el núcleo más ligado por nucleón recorriendo los datos con un ciclo y un condicional, como en Fortran (sin `min`, `np.min` ni `np.argmin`); <br>• una que escriba ese núcleo en `maximo_python.dat`, con el formato de `maximo.dat`; <br>• una que haga la gráfica de $B/A = -E_B$ contra $A$ con los puntos experimentales de la celda C.2 y la guarde como imagen. | P1 a P3 |
| Celda de uso (C.4) | Importa el módulo (las primeras líneas ya están escritas), llama a las cuatro funciones e imprime el núcleo más ligado. | — |

**Resultados de la parte C**: `taller_nucleo_atomico/maximo_python.dat` y `taller_nucleo_atomico/curva_enlace.png`.

**La parte C está terminada cuando** Python imprime el mismo $A$ y $Z$ que `maximo.dat` y la gráfica muestra la curva del modelo con los diez puntos experimentales.

#### C.2 Datos experimentales

Valores de la energía de enlace por nucleón, $B/A = -E_B$, en MeV (evaluación de masas atómicas AME2020). Use esta celda tal como está. Los puntos representan los núclidos indicados; la curva representa el máximo enlace para cada $A$ según el modelo. No son necesariamente el mismo $Z$, por lo que la comparación gráfica es orientativa. Para una comparación numérica de un núclido concreto, use su mismo $Z$ y $A$.

In [ ]:
# Datos experimentales: (símbolo, Z, A, B/A [MeV]) -- AME2020
EXPERIMENTALES = [
    ("He-4",    2,   4, 7.0739),
    ("C-12",    6,  12, 7.6801),
    ("O-16",    8,  16, 7.9762),
    ("Ca-40",  20,  40, 8.5513),
    ("Fe-56",  26,  56, 8.7903),
    ("Ni-62",  28,  62, 8.7945),
    ("Sn-120", 50, 120, 8.5045),
    ("Pb-208", 82, 208, 7.8675),
    ("U-235",  92, 235, 7.5909),
    ("U-238",  92, 238, 7.5701),
]

#### C.3 Módulo de Python

In [ ]:
%%writefile taller_nucleo_atomico/analisis_nucleo.py
"""Taller 6 -- Módulo de análisis (escriba aquí sus cuatro funciones)."""

#### C.4 Uso del módulo

In [ ]:
# C.4 -- Uso del módulo de análisis
# Las tres líneas siguientes ya están escritas: agregan la carpeta del Taller a la
# ruta de búsqueda de Python, importan el módulo y lo releen si fue modificado.
import sys, importlib
sys.path.insert(0, "taller_nucleo_atomico")
import analisis_nucleo
_ = importlib.reload(analisis_nucleo)

# Escriba a continuación las llamadas a sus funciones

### D. Verificación

La celda siguiente revisa los archivos, sus dimensiones, valores numéricos y coherencia. Los resultados de Fortran deben ser posteriores a sus dos archivos fuente. `maximo_python.dat` debe ser posterior al módulo de análisis y a `valle.dat`. Los cuatro controles informan PASS o FAIL sin detener el notebook. **Esta celda no debe modificarse.**

| Caso | Qué se comprueba |
|---|---|
| 1 | Los ocho núclidos de `nucleos.dat`: orden, cuatro términos, $E_B$ y masa atómica aproximada. |
| 2 | `valle.dat` completo: $A=4,\dots,260$, valores enteros válidos y máximo enlace para cada $A$; incluye la referencia $A=208$, $Z=82$. |
| 3 | Referencia $A=238$, $Z=92$, y coherencia entre el $E_B$ guardado y el modelo en todas las filas de `valle.dat`. |
| 4 | Los máximos de Fortran y Python coinciden con el mínimo global de `valle.dat`; existe una imagen PNG no vacía. |

Las referencias se evalúan en doble precisión con las constantes del enunciado. Por ello pueden diferir en los últimos decimales de una salida publicada con otra precisión. Para ${}^{244}$Pu: $E_B \approx -7.53611738$ MeV y $M \approx 227\,342.0074$ MeV.

**4/4 PASS es necesario, pero no suficiente.** La celda no evalúa la arquitectura del código, el uso de `intent`, la independencia de variables locales, la calidad de la gráfica ni las explicaciones. No consulte su implementación como guía para resolver el algoritmo: utilice los mensajes para localizar qué resultado debe revisar.


In [ ]:
# ===================== VERIFICACIÓN — NO MODIFICAR =====================
from pathlib import Path
import numpy as np

CARPETA = Path("taller_nucleo_atomico")
veredictos = []

def informar(nombre, condicion, detalle=""):
    veredictos.append(bool(condicion))
    print(f"{'PASS' if condicion else 'FAIL'} | {nombre}" + (f" — {detalle}" if detalle else ""))

def leer_salida(nombre, filas, columnas, origen="fortran"):
    ruta = CARPETA / nombre
    if not ruta.is_file():
        raise ValueError(f"falta {ruta}")
    if origen == "fortran":
        fuentes = [CARPETA / "modulo_nucleo.f90", CARPETA / "programa_nucleo.f90"]
    else:
        fuentes = [CARPETA / "analisis_nucleo.py", CARPETA / "valle.dat"]
    for fuente in fuentes:
        if not fuente.is_file():
            raise ValueError(f"falta {fuente}")
        if ruta.stat().st_mtime < fuente.stat().st_mtime:
            raise ValueError(f"{nombre} es anterior a {fuente.name}: vuelva a ejecutar el cálculo correspondiente")
    datos = np.loadtxt(ruta, ndmin=2)
    if datos.shape != (filas, columnas) or not np.isfinite(datos).all():
        raise ValueError(f"formato incorrecto en {nombre}: se esperaban {filas} filas y {columnas} columnas finitas")
    return datos

# Implementación de referencia exclusiva de la verificación.
def referencia_terminos(z, a):
    z, a = np.asarray(z, dtype=float), np.asarray(a, dtype=float)
    return np.broadcast_arrays(-15.8 + np.zeros_like(a), 18.3*a**(-1.0/3.0),
                               0.71*z**2*a**(-4.0/3.0), 92.7*(z-a/2.0)**2/a**2)

def referencia_enlace(z, a):
    return sum(referencia_terminos(z, a))

def leer_valle():
    datos = leer_salida("valle.dat", 257, 3)
    if not np.array_equal(datos[:, 0], np.arange(4, 261)):
        raise ValueError("valle.dat debe tener A = 4, 5, ..., 260, en ese orden")
    if not np.array_equal(datos[:, 1], np.rint(datos[:, 1])):
        raise ValueError("los valores de Z deben ser enteros")
    if np.any((datos[:, 1] < 1) | (datos[:, 1] > datos[:, 0])):
        raise ValueError("cada fila debe cumplir 1 <= Z <= A")
    return datos

# 1) Ocho núclidos: todos los términos, el enlace y la masa.
try:
    nuc = leer_salida("nucleos.dat", 8, 8)
    orden = np.array([(94,244), (32,70), (72,179), (18,38), (26,56), (20,40), (92,235), (92,238)])
    if not np.array_equal(nuc[:, :2], orden):
        raise ValueError("Z y A deben ser enteros y seguir el orden del enunciado")
    z, a = orden[:, 0], orden[:, 1]
    terminos = np.column_stack(referencia_terminos(z, a))
    eb = terminos.sum(axis=1)
    masa = z*938.78 + (a-z)*939.57 + a*eb
    ok = (np.allclose(nuc[:, 2:6], terminos, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 6], eb, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 7], masa, rtol=0, atol=1e-3)
          and np.allclose(nuc[:, 2:6].sum(axis=1), nuc[:, 6], rtol=0, atol=1e-7))
    informar("Caso 1: ocho núclidos", ok, f"Pu-244: E_B = {nuc[0,6]:.8f} MeV; M = {nuc[0,7]:.4f} MeV")
except Exception as error:
    informar("Caso 1: ocho núclidos", False, str(error))

# 2) La búsqueda obtiene el máximo enlace para cada A del intervalo.
try:
    valle = leer_valle()
    correctos = []
    for a, z, eb in valle:
        candidatos = np.arange(1, int(a)+1)
        energias = referencia_enlace(candidatos, a)
        correctos.append(abs(float(referencia_enlace(z, a)) - energias.min()) <= 1e-7)
    fila = valle[valle[:, 0] == 208][0]
    ok = all(correctos) and fila[1] == 82 and abs(fila[2] - (-7.800561449221129)) <= 1e-7
    informar("Caso 2: máximo enlace para cada A", ok, f"{sum(correctos)}/257 búsquedas correctas; A=208: Z={fila[1]:.0f}")
except Exception as error:
    informar("Caso 2: máximo enlace para cada A", False, str(error))

# 3) Energías de todas las filas y referencia de U-238.
try:
    valle = leer_valle()
    esperado = referencia_enlace(valle[:, 1], valle[:, 0])
    fila = valle[valle[:, 0] == 238][0]
    ok = (np.allclose(valle[:, 2], esperado, rtol=0, atol=1e-7)
          and fila[1] == 92 and abs(fila[2] - (-7.579586363043683)) <= 1e-7)
    informar("Caso 3: energías de la curva y U-238", ok, f"A=238: Z={fila[1]:.0f}; E_B={fila[2]:.8f} MeV")
except Exception as error:
    informar("Caso 3: energías de la curva y U-238", False, str(error))

# 4) Mínimo global: dos lenguajes y archivo de imagen.
try:
    valle = leer_valle()
    mf = leer_salida("maximo.dat", 1, 3)[0]
    mp = leer_salida("maximo_python.dat", 1, 3, origen="python")[0]
    referencia = valle[np.argmin(valle[:, 2])]
    def coincide(fila):
        return (np.array_equal(fila[:2], referencia[:2])
                and abs(fila[2]-referencia[2]) <= 1e-7)
    imagen = CARPETA / "curva_enlace.png"
    png_ok = (imagen.is_file() and imagen.stat().st_size > 8
              and imagen.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")
    ok = (coincide(mf) and coincide(mp)
          and abs(referencia[2] - (-8.69506905766211)) <= 1e-7 and png_ok)
    informar("Caso 4: mínimo global e imagen", ok,
             f"Fortran: A={mf[0]:.0f}, Z={mf[1]:.0f}; Python: A={mp[0]:.0f}, Z={mp[1]:.0f}; PNG: {'presente' if png_ok else 'falta o formato incorrecto'}")
except Exception as error:
    informar("Caso 4: mínimo global e imagen", False, str(error))

print(f"\nResumen: {sum(veredictos)}/4 PASS")
print("La arquitectura, los requisitos y la interpretación se revisan por separado.")


### E. Preguntas de interpretación

Responda en celdas de texto, debajo de cada pregunta. Use los resultados de su propio programa. Los cálculos adicionales de esta sección pueden realizarse llamando a los procedimientos de sus módulos; **no cambie el modelo de cuatro términos ni los archivos que utiliza la verificación de D**.

1. Usando `nucleos.dat`, elija un núcleo relativamente liviano y otro pesado e identifíquelos. Compare los términos de superficie y Coulomb en ambos casos. ¿Cuál de esos dos términos penaliza más el enlace en cada uno? Explique cómo su competencia contribuye a que la curva $B/A$ presente un máximo.

*Escriba aquí su respuesta y los resultados que la sustentan.*

2. El modelo predice su mayor enlace por nucleón en un $A$ distinto del ${}^{62}$Ni experimental. Calcule la diferencia de $E_B$ entre el máximo del modelo y el punto de su curva con $A=62$. Exprese también la diferencia como porcentaje de la magnitud de $E_B$ en ese punto. Discuta el tamaño de la diferencia y las limitaciones del modelo que podrían contribuir a ella. Distinga esta comparación entre puntos de la curva de una comparación directa con el núclido experimental: para esta última debe usar $Z=28$, $A=62$.

*Escriba aquí su respuesta y los resultados que la sustentan.*

3. Para $A=56$, ¿qué $Z$ tiene el mayor enlace por nucleón en el modelo de cuatro términos? Compárelo con el hierro ($Z=26$): calcule la diferencia de energía **total**, indicando el orden de la resta, y diga cuál resulta más ligado.

   La fórmula completa incluye un quinto término, el de **apareamiento**: protones y neutrones tienden a formar pares. En la energía total, la corrección vale $-a_P A^{-1/2}$ si $Z$ y $N$ son ambos pares, $+a_P A^{-1/2}$ si ambos son impares, y cero si $A$ es impar, con $a_P\approx12$ MeV. Calcule la corrección para los dos núcleos comparados y determine si invierte su orden de enlace. Recuerde que esta corrección es de energía **total**: para sumarla a $E_B$ debe dividirla entre $A$.

   Este es un **cálculo adicional de comparación**: no incorpore el apareamiento a las funciones del modelo base ni regenere con él los archivos de D. Basta comparar los dos candidatos; no se pide repetir la búsqueda global con el modelo ampliado.

*Escriba aquí su respuesta y los resultados que la sustentan.*

4. Reutilizando las funciones de su módulo, calcule la energía liberada en la fisión simétrica idealizada ${}^{236}\mathrm{U}\to2\,{}^{118}\mathrm{Pd}$ ($Z=46$): $Q=M(92,236)-2M(46,118)$. Las masas se utilizan aquí como energías equivalentes en MeV. Como ambos lados tienen el mismo número total de protones, las masas electrónicas incluidas se cancelan en esta diferencia, dentro de la aproximación adoptada. Si $Q>0$, el canal es energéticamente favorable. Compárelo con la escala típica de unos 200 MeV por fisión del uranio y explique por qué este canal idealizado no reproduce todos los productos ni el reparto de energía de una fisión real.

*Escriba aquí su respuesta y los resultados que la sustentan.*

### F. Entregables y reporte

**En el repositorio GitHub**, en la carpeta `Taller_6/`: este notebook ejecutado de principio a fin, con el pseudocódigo marco completado en la sección B.1, y la carpeta `taller_nucleo_atomico/` con `modulo_nucleo.f90`, `programa_nucleo.f90`, `analisis_nucleo.py`, los cuatro archivos `.dat` y `curva_enlace.png`.

**En el Campus Virtual**: el reporte de una página, con la plantilla LaTeX del curso, y el enlace a la carpeta del repositorio. El reporte debe incluir:

- **Contexto físico:** qué describe la fórmula semiempírica y qué representa la curva de máximo enlace calculada, distinguiéndola de la estabilidad frente al decaimiento.
- **Método:** cómo se organizó el programa (módulo, funciones, subrutina), apoyado en su pseudocódigo, y cómo se buscó el $Z$ de máximo enlace.
- **Resultado:** la gráfica `curva_enlace.png` y la comparación del modelo con dos valores experimentales, usando para cada comparación el mismo $Z$ y $A$. Indique las diferencias numéricas y la convención de signos.
- **Obstáculo o decisión de implementación:** por ejemplo, el orden de compilación, el uso de `intent` o el paso de enteros a reales en las potencias $A^{-1/3}$.
- **Conclusión:** incluya la respuesta a la pregunta E3.
- **Uso de herramientas de IA:** cuáles se usaron y para qué, si se usó alguna.

**Criterios de revisión**

Se valorarán la organización en procedimientos, el uso de argumentos e `intent`, las variables locales, la reutilización de funciones y la separación entre cálculo, salida y análisis. También se revisarán los resultados numéricos, la claridad de la gráfica, la interpretación física y la capacidad de explicar y modificar el propio código en la sustentación. Los PASS no reemplazan estos criterios.

**Antes de entregar**

- [ ] Los tres espacios del pseudocódigo marco (sección B.1) están completos.
- [ ] Reiniciar y ejecutar todas las celdas completa el flujo; solo aparecen los errores de compilación anunciados en los ejemplos didácticos. El código del Taller 6 compila y se ejecuta sin errores.
- [ ] La verificación de la sección D da 4/4 PASS.
- [ ] El código de Fortran cumple los requisitos R1 a R7, y el de Python, P1 a P3.
- [ ] Cada procedimiento tiene un comentario que dice qué calcula, con qué entradas y qué salidas.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** entregar archivos `.dat` de una ejecución anterior: la verificación compara su fecha con la del código y marca FAIL si el código cambió después. La comprobación también revisa que `maximo_python.dat` no sea anterior al módulo de Python ni a `valle.dat`. Antes de entregar, reinicie el entorno y ejecute todas las celdas (en Colab: reiniciar sesión y ejecutar todo; en Jupyter: `Restart & Run All`). Los ejemplos de errores anunciados son deliberados.

</div>